# Workflow with Tools — Build & Trigger

This notebook combines two earlier ones:

- [`02_interactive_workflow.ipynb`](02_interactive_workflow.ipynb) — building a workflow and chatting with it
- [`04_tools_and_inbuilt.ipynb`](04_tools_and_inbuilt.ipynb) — tools that LLM nodes can call

Here we **attach tools to an LLM node** and watch them fire during a live conversation:

1. Define two inline-Python tools with `ToolsConfig` — each looks something up in **private data the model has never seen**
2. Build a `SayLLMNodeConfig` that carries those tools via `tools_config`
3. Chat interactively and observe the **tool call -> tool result -> answer** sequence in the events
4. Inspect the raw events to prove the tools were triggered

> **Why these tools?** A tool only fires when the model believes it *cannot* answer on its own. Give it
> arithmetic (`add_numbers`, `order_total`) and it will just compute the answer itself and skip the tool.
> So both tools here are **lookups into records the model has no way to know** — an order-status table and
> an inventory table — which reliably forces genuine tool use.

> **Tip:** configs are built with the typed classes from `interactly.configs` (requires `pip install "interactly[configs]"`).

> **These notebooks are async-first.** They use `AsyncWorkflowClient` with top-level `await`,
> which runs directly in Jupyter (the setup cell calls `nest_asyncio.apply()`). Every method
> shown also exists on the synchronous `WorkflowClient` — just drop the `await`. See the
> [docs](../docs/README.md) for the sync surface. Notebook bodies stay 100% async — there is
> no per-notebook sync cell.

In [ ]:
import _bootstrap  # noqa: F401 - enables import interactly (no install needed)

import os, sys
import nest_asyncio
from dotenv import load_dotenv
from pathlib import Path

# This is required to run asyncio in Jupyter Notebook
nest_asyncio.apply()

# Get the current notebook directory and find the project root
current_dir = Path(os.getcwd())
project_root = current_dir
while project_root.parent != project_root:
    if (project_root / '.env').exists():
        break
    project_root = project_root.parent
else:
    project_root = current_dir
    for _ in range(5):
        if (project_root / 'pyproject.toml').exists():
            break
        project_root = project_root.parent

# Load the environment variables from .env in project root
env_path = project_root / '.env'
if env_path.exists():
    load_dotenv(dotenv_path=str(env_path))
    print(f"Loaded .env from: {env_path}")
else:
    print(f"Warning: .env file not found at {env_path}")

# Add the project root to sys.path so imports resolve
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
    print(f"Added to Python path: {project_root}")
else:
    print(f"Project root already in Python path: {project_root}")

In [ ]:
# Interactly credentials are read from environment variables.
#
# Convenience defaults point at the dev "Workflow Illustrations" org; your shell
# environment always wins (setdefault only fills in what you have not set).
os.environ.setdefault("INTERACTLY_BASE_URL", "https://api-dev.interactly.ai/workflows")
os.environ.setdefault("INTERACTLY_TEAM_ID", "67458e762b7d3dc15aaea5b5")
os.environ.setdefault("INTERACTLY_USER_ID", "687b1a4f745c8e6806c98d91")

# The bearer token is a secret — never hardcode it in the notebook.
# Export it before launching Jupyter:  export INTERACTLY_API_KEY="…"
assert os.environ.get("INTERACTLY_API_KEY"), (
    "Set INTERACTLY_API_KEY in your environment before running this notebook."
)

#print(f"API KEY is: {os.getenv('INTERACTLY_API_KEY')}")
print(f"TEAM ID is: {os.getenv('INTERACTLY_TEAM_ID')}")
print(f"USER ID is: {os.getenv('INTERACTLY_USER_ID')}")
print(f"BASE URL is: {os.getenv('INTERACTLY_BASE_URL')}")

In [ ]:
import _bootstrap  # noqa: F401 - enables import interactly (no install needed)

import nest_asyncio

from interactly import AsyncWorkflowClient

# Required to run top-level `await` inside a Jupyter notebook
nest_asyncio.apply()

client = AsyncWorkflowClient()
print("Connected to", client._base_url)

## 1. Define the tools

An LLM node calls tools that live in its `tools_config`. We use `InlinePythonToolConfig`,
which carries a `name`, a `signature` (the description the model sees), an `args_schema`
(JSON Schema for the arguments), and the executable `code`. The runtime executes the
Python and stores each tool's return value in a runtime variable (`result_runtime_variable_name`,
default `tool_result`).

Both tools below take a **key** (an order id, a product SKU) and return the value looked up in
a small in-function table. Because that data lives only inside the tool — not in the model's
training — the assistant has to call the tool to answer.

In [ ]:
from interactly.configs import ToolsConfig, InlinePythonToolConfig

# Tool 1: look up the status of an order by its id.
# These order records live only inside the function — the model has never
# seen them, so it cannot answer without calling this tool.
order_status_tool = InlinePythonToolConfig(
    name="lookup_order_status",
    signature="Look up the current status and delivery estimate of an order, given its order id (e.g. 'ORD-1002').",
    args_schema={
        "type": "object",
        "properties": {
            "order_id": {"type": "string", "description": "The order identifier, e.g. 'ORD-1002'"},
        },
        "required": ["order_id"],
    },
    code=(
        "def lookup_order_status(order_id: str) -> dict:\n"
        "    orders = {\n"
        "        'ORD-1001': {'status': 'Delivered', 'carrier': 'UPS', 'eta': '2026-07-02'},\n"
        "        'ORD-1002': {'status': 'In transit', 'carrier': 'FedEx', 'eta': '2026-07-12'},\n"
        "        'ORD-1003': {'status': 'Processing', 'carrier': None, 'eta': '2026-07-15'},\n"
        "        'ORD-1004': {'status': 'Cancelled', 'carrier': None, 'eta': None},\n"
        "    }\n"
        "    return orders.get(order_id.strip().upper(), {'status': 'No order found with that id'})"
    ),
)

# Tool 2: check how many units of a product (by SKU) are in stock, and where.
inventory_tool = InlinePythonToolConfig(
    name="check_inventory",
    signature="Check how many units of a product are in stock and at which warehouse, given its SKU (e.g. 'WIDGET-BLUE').",
    args_schema={
        "type": "object",
        "properties": {
            "sku": {"type": "string", "description": "The product SKU, e.g. 'WIDGET-BLUE'"},
        },
        "required": ["sku"],
    },
    code=(
        "def check_inventory(sku: str) -> dict:\n"
        "    stock = {\n"
        "        'WIDGET-BLUE': {'in_stock': 42, 'warehouse': 'Austin, TX'},\n"
        "        'WIDGET-RED': {'in_stock': 0, 'warehouse': 'Reno, NV'},\n"
        "        'GADGET-PRO': {'in_stock': 7, 'warehouse': 'Newark, NJ'},\n"
        "        'GADGET-LITE': {'in_stock': 128, 'warehouse': 'Austin, TX'},\n"
        "    }\n"
        "    return stock.get(sku.strip().upper(), {'in_stock': 0, 'warehouse': 'No product found with that SKU'})"
    ),
)

support_tools = ToolsConfig(tools=[order_status_tool, inventory_tool])
print(f"Configured {len(support_tools.tools)} tools: " + ", ".join(t.name for t in support_tools.tools))

## 2. Build a workflow whose LLM node carries the tools

The assistant node is a `SayLLMNodeConfig` with `tools_config=support_tools`. Its prompt
tells it that it has **no order or inventory data of its own**, so it must call a tool to
answer any such question (this is what makes the tools reliably fire).
`self_loop=True` keeps the node active across turns, and a conditional edge routes to an
end node when the user is done. Creating the workflow publishes an active `v0`.

In [ ]:
from interactly.configs import (
    SayLLMNodeConfig,
    SayStaticMessageNodeConfig,
    PromptConfig,
    StaticMessagesConfig,
    ConditionalEdgeConfig,
    ConditionConfig,
    WorkflowConfig,
    WorkflowConfigFullyHydrated,
    OpenAILLMConfig,
    OPENAIModel,
)
from interactly.types.workflows.workflow import Workflow

ASSISTANT_PROMPT = (
    "You are a helpful order-support agent.\n"
    "- You have NO order or inventory data memorized. You must NEVER guess an order status, "
    "delivery date, or stock level from memory.\n"
    "- To answer any question about an order, you MUST call lookup_order_status with the order id "
    "the user gives (e.g. 'ORD-1002').\n"
    "- To answer any question about whether a product is available, you MUST call check_inventory "
    "with the product SKU the user gives (e.g. 'WIDGET-BLUE').\n"
    "- After a tool returns, state the result in one friendly sentence and ask if there is anything else."
)

assistant_node = SayLLMNodeConfig(
    name="Support Assistant",
    is_start=True,
    self_loop=True,
    wait_for_user_message=True,
    main_response_config=PromptConfig(prompt=ASSISTANT_PROMPT),
    llms_config=OpenAILLMConfig(model=OPENAIModel.GPT_5_4_MINI, max_tokens=400),
    tools_config=support_tools,
    max_consecutive_tool_calls=3,  # allow a couple of tool calls per turn
)

end_node = SayStaticMessageNodeConfig(
    name="End",
    static_messages_config=StaticMessagesConfig(static_messages=["Thanks for chatting - goodbye!"]),
    wait_for_user_message=False,
)

config = WorkflowConfigFullyHydrated(
    workflow_config=WorkflowConfig(
        name="Order Support Agent with Tools",
        description="LLM node wired to inline-Python lookup tools (05_workflow_with_tools.ipynb).",
    ),
    node_configs=[assistant_node, end_node],
    edge_configs=[
        ConditionalEdgeConfig(
            source_node_logical_id=assistant_node.logical_id,
            destination_node_logical_id=end_node.logical_id,
            name="Assistant -> End",
            condition=ConditionConfig(
                condition_freeform="The user wants to end the conversation or says goodbye."
            ),
        )
    ],
)

workflow: Workflow = await client.workflows.create_from_config(config)
WF_ID = workflow.id
print(f"Created workflow  id={WF_ID}  name={workflow.name!r}")

## 3. Set up an interactive chat that surfaces tool activity

We drive turns with an `AsyncWorkflowHandle` (it tracks the `run_id` for us). Tool activity
shows up inside `SayLLMNodeEvent` events:

- an **AI** message with a non-empty `tool_calls` list -> the model is calling a tool
- a **tool** message (`type == "tool"`) -> the tool's return value

The final natural-language reply arrives as an `AssistantResponseEvent`. The helper below
prints each of these as a labelled line and returns the raw events for inspection.

> Conditional edges also route via an internal tool-call mechanism (names starting with
> `Destination_Node...`); we filter those out so only *your* tools are shown.

In [ ]:
from langchain_core.messages import HumanMessage

from interactly.runtime.handle import AsyncWorkflowHandle
from interactly.runtime.events import (
    AssistantResponseEvent,
    BusyWaitForUserMessageEvent,
    SayLLMNodeEvent,
)
from interactly.configs import (
    LLMNodeRunInput,
    NodesRunInputs,
    WorkflowCommand,
    WorkflowRunInput,
)

chat: AsyncWorkflowHandle = await client.workflows.handle(WF_ID)


async def send_message(user_text: str, *, command: WorkflowCommand = WorkflowCommand.DATA):
    """Send one user turn, print assistant replies and tool activity, and return the raw events."""
    print(f"User: {user_text}")

    run_input = WorkflowRunInput(
        command=command,
        thread_to_node_inputs={
            "0": NodesRunInputs(
                node_run_inputs=[LLMNodeRunInput(messages=[HumanMessage(content=user_text)])]
            )
        },
    )

    events = []
    async for event in chat.arun(run_input):
        events.append(event)
        if isinstance(event, SayLLMNodeEvent) and isinstance(event.message, dict):
            msg = event.message
            for call in msg.get("tool_calls") or []:
                name = call.get('name') or ''
                # Skip internal edge-routing calls (they are not user tools)
                if name.startswith("Destination_Node"):
                    continue
                print(f"  \U0001F527 tool call: {name}({call.get('args')})")
            if msg.get("type") == "tool":
                result = (msg.get("response_metadata") or {}).get("tool_result", msg.get("content"))
                print(f"  \u2705 tool result: {result}")
        elif isinstance(event, AssistantResponseEvent) and event.content:
            print(f"  Assistant: {event.content}")
        elif isinstance(event, BusyWaitForUserMessageEvent):
            print("  (waiting for the next user message)")
    return events

## 4. Turn 1 — trigger the `lookup_order_status` tool

The first turn uses `WorkflowCommand.START`. Ask about a specific order id. The model has no
way to know the status, so it calls the tool and reports what comes back.

In [ ]:
turn1_events = await send_message("Can you check the status of my order ORD-1002?", command=WorkflowCommand.START)

## 5. Turn 2 — trigger the `check_inventory` tool

Ask whether a product (by SKU) is in stock. Again the model can't know, so it calls the
inventory tool and relays the count.

In [ ]:
turn2_events = await send_message("Do you have any WIDGET-BLUE in stock right now?")

## 6. Inspect the raw tool events

To prove the tools actually ran, pull the tool call(s) and tool result(s) out of the
second turn's events.

In [ ]:
def extract_tool_activity(events):
    calls, results = [], []
    for event in events:
        if isinstance(event, SayLLMNodeEvent) and isinstance(event.message, dict):
            msg = event.message
            for call in msg.get("tool_calls") or []:
                name = call.get("name") or ''
                if name.startswith("Destination_Node"):
                    continue  # skip internal edge-routing calls
                calls.append((name, call.get("args")))
            if msg.get("type") == "tool":
                results.append((msg.get("response_metadata") or {}).get("tool_result", msg.get("content")))
    return calls, results


calls, results = extract_tool_activity(turn2_events)
print("Tool calls made this turn:")
for name, args in calls:
    print(f"  - {name}({args})")
print("Tool results returned:")
for r in results:
    print(f"  - {r}")

## 7. End the conversation

Saying goodbye satisfies the conditional edge, routing to the end node.

In [ ]:
end_events = await send_message("That's everything, thanks. Goodbye!")
print(f"\nSession run_id: {chat.run_id}")

## 8. Cleanup

In [ ]:
await client.workflows.delete(WF_ID)
print(f"Workflow {WF_ID} deleted.")

await client.close()

## See also

- [`04_tools_and_inbuilt.ipynb`](04_tools_and_inbuilt.ipynb) — the full tool catalogue and custom-tool CRUD
- [`02_interactive_workflow.ipynb`](02_interactive_workflow.ipynb) — building a workflow and the chat handle
- Guide: [`../docs/guides/nodes_edges_tools.md`](../docs/guides/nodes_edges_tools.md)